# Team 3 — TTCC · URA Hackathon 2026 


## Cell 1 — Setup

In [ ]:
import os
os.environ["FLAGS_use_mkldnn"]         = "0"
os.environ["FLAGS_enable_onednn"]      = "0"
os.environ["DNNL_DEFAULT_FPMATH_MODE"] = "STRICT"
os.environ["PADDLE_DISABLE_MKL"]       = "1"
os.environ["OMP_NUM_THREADS"]          = "2"

!pip install "paddlepaddle==2.6.2" -q
!pip install "paddleocr>=2.9,<3.0"  -q
!pip install tqdm scikit-learn Pillow -q

import paddle, paddleocr
print(f"paddlepaddle : {paddle.__version__}    (requires 2.6.2)")
print(f"paddleocr    : {paddleocr.__version__}  (requires <3.0)")
print("Setup complete \u2713")


## Cell 2 — Load Data

In [ ]:
import re, time, zipfile, os
import pandas as pd
from pathlib import Path

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from PIL import Image, ImageEnhance

def locate_dataset():
    root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path(".")

    test_csv = None
    for csv_name in ["private_test.csv", "test.csv"]:
        found = list(root.rglob(csv_name))
        if found:
            test_csv = found[0]
            break

    if not test_csv:
        raise FileNotFoundError("Could not find private_test.csv or test.csv in the dataset.")

    input_dir = test_csv.parent

    images_dir = None
    valid_exts = {".jpg", ".jpeg", ".png"}

    for search_base in [input_dir, input_dir.parent]:
        if not search_base.exists(): continue
        for p in search_base.rglob("*"):
            if p.is_dir() and p.name in ["images", "test_images"]:
                if any(f.suffix.lower() in valid_exts for f in p.iterdir() if f.is_file()):
                    images_dir = p
                    break
        if images_dir:
            break

    if not images_dir:
        for zname in ["test_images.zip", "images.zip"]:
            for p in root.rglob(zname):
                er = Path("/kaggle/working/dataset_images")
                er.mkdir(exist_ok=True, parents=True)
                with zipfile.ZipFile(p) as zf:
                    zf.extractall(er)
                for ext_dir in er.rglob("*"):
                    if ext_dir.is_dir() and any(f.suffix.lower() in valid_exts for f in ext_dir.iterdir() if f.is_file()):
                        images_dir = ext_dir
                        break
                if images_dir: break
            if images_dir: break

    if not images_dir:
        raise FileNotFoundError(f"Found {test_csv.name} but could NOT find an images directory containing .jpg/.png.")

    train_labels_csv = None
    found_labels = list(root.rglob("train_labels.csv"))
    if found_labels:
        train_labels_csv = found_labels[0]

    return input_dir, test_csv, images_dir, train_labels_csv

INPUT_DIR, TEST_CSV, IMAGES_DIR, TRAIN_LABELS_CSV = locate_dataset()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

SAMPLE_CSV = INPUT_DIR / "sample_submission_private.csv" if (INPUT_DIR / "sample_submission_private.csv").exists() else INPUT_DIR / "sample_submission.csv"
OUTPUT_CSV = WORK_DIR / "submission.csv"
CHECKPOINT_CSV = WORK_DIR / "checkpoint.csv"

test_df = pd.read_csv(TEST_CSV)
on_disk = {p.stem for p in IMAGES_DIR.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]}

train_labels_df = None
if TRAIN_LABELS_CSV and TRAIN_LABELS_CSV.exists():
    train_labels_df = pd.read_csv(TRAIN_LABELS_CSV, keep_default_na=False)

print(f"Input dir   : {INPUT_DIR}")
print(f"Images dir  : {IMAGES_DIR}  ({len(on_disk):,} images)")
print(f"Test file   : {TEST_CSV.name} ({len(test_df):,} images)")
print(f"Missing     : {len(set(test_df['image_id']) - on_disk)}")

if train_labels_df is not None:
    of = (train_labels_df["ocr_text"].astype(str).str.strip()!="").mean()
    pf = (train_labels_df["product_name"].astype(str).str.strip()!="").mean()
    print(f"Train labels: {len(train_labels_df):,} rows | OCR {of:.1%} | product {pf:.1%}")
else:
    print("Train labels: not found (model will run entirely on rules)")


## Cell 3 — Brand Rules: Brand_name & Product_name (separated)


In [ ]:
import re
from unicodedata import normalize as _unorm

def _strip_accent(s: str) -> str:
    s = _unorm('NFD', s)
    s = re.sub(r'[\u0300-\u036f]', '', s)
    return _unorm('NFC', s.replace('đ','d').replace('Đ','D')).lower()

BRAND_RULES = [
    (r'ha.?long.{0,25}canf[uo]c|canf[uo]c.{0,25}ha.?long|halongcanf',
     'Ha Long Canfoco'),
    (r'\bcanfoco\b|\bcanfood\b|canf[uo]co\b',
     'Ha Long Canfoco'),
    (r'(cong\s*ty|ctcp|co\s*phan|cty).{0,50}(do\s*hop|đồ\s*hộp).{0,30}ha.?long',
     'Đồ Hộp Hạ Long'),
    (r'(do\s*hop|đồ\s*hộp).{0,30}ha.?long',
     'Đồ Hộp Hạ Long'),
    (r'ha.?long.{0,30}(do\s*hop|đồ\s*hộp)',
     'Đồ Hộp Hạ Long'),
    (r'pate.{0,15}(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn)|patecot(?:den)?',
     'Pate Cột Đèn Hải Phòng'),
    (r'(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn).{0,20}(hai\s*phong|hải\s*phòng)',
     'Pate Cột Đèn Hải Phòng'),
    (r'ha.?long|halong',
     'Đồ Hộp Hạ Long'),

    (r'nan.{0,5}optipro.{0,5}plus|nan.{0,5}optiproplus',
     'Nestlé NAN OPTIpro PLUS'),
    (r'nan.{0,5}infinipro|nan.{0,5}infini\b',
     'Nestlé NAN INFINIPRO A2'),
    (r'nan.{0,5}supremepro|nan.{0,5}supreme',
     'Nestlé NAN SUPREMEpro'),
    (r'nan.{0,5}optipro\b',
     'Nestlé NAN OPTIpro'),
    (r'\bnan\b',
     'Nestlé NAN'),
    (r'\bbeba\b',
     'Nestlé BEBA'),
    (r'nestl[eé].{0,10}alfamino|\balfamino\b',
     'Nestlé Alfamino'),

    (r'\bmilo\b',                  'Nestlé Milo'),
    (r'nestle|nestlé',               'Nestlé'),
    (r'vinamilk',                    'Vinamilk'),
    (r'th\s*true|thtrue',           'TH True Milk'),
    (r'dutch\s*lady',                'Dutch Lady'),
    (r'\baptamil\b',               'Aptamil'),
    (r'similac',                     'Abbott Similac'),
    (r'\bensure\b',                'Abbott Ensure'),
    (r'pediasure',                   'Abbott PediaSure'),
    (r'glucerna',                    'Abbott Glucerna'),
    (r'\bfriso\b',                 'Friso'),
    (r'\bhipp\b',                  'HiPP'),
    (r'nutifood|\bnuti\b',         'Nutifood'),
    (r'optimum\s*gold',             'Optimum Gold'),
    (r'\bmeiji\b',                 'Meiji'),
    (r'\billuma\b',                'Illuma'),
    (r'\banlene\b',                'Anlene'),
    (r'\byomost\b',                'Yomost'),
    (r'\bfami\b',                  'Fami'),
    (r'lothamilk',                   'Lothamilk'),
    (r'ba\s*v[iì]\b|\bbavi\b',    'Ba Vì'),
    (r'dalat\s*milk',               'Đà Lạt Milk'),
    (r'\bkun\b',                   'Kun'),

    (r'highlands?.{0,5}coffee|\bhighlands\b', 'Highlands Coffee'),
    (r'the\s*coffee\s*house|coffee\s*house', 'Coffee House'),
    (r'phuc\s*long|phúc\s*long',              'Phúc Long'),

    (r'\bvissan\b',                'Vissan'),
    (r'chin[\s\-]*su|chinsu',       'Chin-Su'),
    (r'quang\s*h[oô]ng',            'Quang Hong Sardine'),
    (r'\bhafi\b',                  'Hafi'),
    (r'sardine|cá\s*mòi',           'Sardine'),
    (r'pate\s*minh\s*chay|minh\s*chay', 'Pate Minh Chay'),

    (r'\bpate\b|\bpatê\b',
     'Pate Cột Đèn Hải Phòng'),
]

LINE_RULES = {
    'Ha Long Canfoco': [
        (r'pate.{0,10}(c[oộ]t|cot).{0,10}(d[eèẻ]n|đèn)|patecot', 'Pate Cột Đèn'),
    ],
    'Vinamilk': [
        (r'\bflex\b',            'Flex'),
        (r'adm\s*gold',           'ADM Gold'),
        (r'\bsure\b',            'Sure'),
        (r'colosbaby',             'ColosBaby'),
        (r'dielac',                'Dielac'),
        (r'\bgrow\b',            'Grow'),
        (r'\bcanxi\b',           'Canxi'),
        (r'ong\s*tho|ông\s*thọ', 'Ông Thọ'),
        (r'sua\s*chua|sữa\s*chua', 'Sữa Chua'),
    ],
    'Nestlé Milo': [
        (r'3\s*in\s*1|3in1',     '3in1'),
    ],
    'TH True Milk': [
        (r'true\s*yogurt',        'True Yogurt'),
        (r'\bgrow\b',            'Grow'),
        (r'school\s*milk',        'School Milk'),
        (r'true\s*butter',        'True Butter'),
    ],
    'Dutch Lady': [
        (r'grow\s*\+?|grow\s*plus', 'Grow+'),
        (r'complete',              'Complete'),
        (r'\bcanxi\b',           'Canxi'),
        (r'\bmom\b',             'Mum'),
    ],
    'Vissan': [
        (r'pate\s*heo',           'Pate Heo'),
        (r'pate\s*g[aà]\b',      'Pate Gà'),
        (r'xuc\s*xich|xúc\s*xích', 'Xúc Xích'),
        (r'cá\s*x[oố]t\s*c[aà]', 'Cá Xốt Cà'),
    ],
    'Highlands Coffee': [
        (r'tr[aà]\s*sen\s*v[aà]ng', 'Trà Sen Vàng'),
        (r'tr[aà]\s*v[aả]i',         'Trà Vải'),
        (r'americano',                'Americano'),
        (r'phuc\s*kien|phúc\s*kiến','Phúc Kiến'),
    ],
    'Nutifood': [
        (r'growplus|grow\s*plus', 'GrowPlus'),
        (r'\bpedia\b',           'Pedia'),
    ],
    'Aptamil': [
        (r'profutura',             'Profutura'),
    ],
    'Friso': [
        (r'\bgold\b',            'Gold'),
        (r'comfort',               'Comfort'),
        (r'prestige',              'Prestige'),
    ],
    'Abbott Ensure': [
        (r'\bgold\b',            'Gold'),
    ],
    'Anlene': [
        (r'\bgold\b',            'Gold'),
        (r'concentrate',           'Concentrate'),
    ],
    'Ba Vì': [
        (r'\bgold\b',            'Gold'),
    ],
    'Kun': [
        (r'chocolate',             'Chocolate'),
        (r'strawberry',            'Strawberry'),
    ],
}

_BRAND_HAS_BUILTIN_LINE = {
    'Nestlé NAN OPTIpro PLUS', 'Nestlé NAN INFINIPRO A2',
    'Nestlé NAN SUPREMEpro', 'Nestlé NAN OPTIpro',
    'Pate Cột Đèn Hải Phòng', 'Đồ Hộp Hạ Long',
}

NOISE_PRODUCTS = {
    'NS RECORDS', 'Bình trữ sữa', 'CapCut', 'TikTok',
    'Finance', 'ChatGPT', 'LINH IT', 'CP',
}

BRAND_NORMALIZE_MAP = [
    (r'^ha long canfood$',               'Ha Long Canfoco'),
    (r'^nan$|^sữa nan$',                 'Nestlé NAN'),
    (r'^nestle beba$',                   'Nestlé BEBA'),
]


def _find_brand(tl: str, tls: str) -> str:
    for pattern, brand in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE) or re.search(pattern, tls, re.IGNORECASE):
            return brand
    return ''


def _find_line(brand: str, tl: str, tls: str) -> str:
    for pattern, line in LINE_RULES.get(brand, []):
        if re.search(pattern, tl, re.IGNORECASE) or re.search(pattern, tls, re.IGNORECASE):
            return line
    return ''


def extract_brand_product(text: str) -> tuple:
    """Returns (brand_name, product_name) separately per the new spec.

    brand_name   : canonical brand name (e.g. 'Vinamilk', 'Highlands Coffee')
    product_name : product line ONLY, brand is not merged in
                   (e.g. 'Flex', 'Trà Sen Vàng'); empty if no line matches.
    """
    if not text or not text.strip():
        return '', ''

    clean = re.sub(
        r'(công\s*ty|cty|ctcp|cổ\s*phần|tổng\s*giám\s*đốc|giám\s*đốc)\s*cp\b',
        'COTYCP', text, flags=re.IGNORECASE
    )
    tl  = clean.lower()
    tls = _strip_accent(clean)

    brand = _find_brand(tl, tls)
    if not brand:
        return '', ''

    line = ''
    if brand not in _BRAND_HAS_BUILTIN_LINE:
        line = _find_line(brand, tl, tls)

    return brand, line


def normalize_brand(name: str) -> str:
    if not name: return ''
    name = name.strip()
    for pat, canon in BRAND_NORMALIZE_MAP:
        if re.fullmatch(pat, name, re.IGNORECASE):
            return canon
    return name


def normalize_product(name: str) -> str:
    if not name: return ''
    name = name.strip()
    if name in NOISE_PRODUCTS: return ''
    name = re.sub(r'#\S+', '', name).strip()
    name = re.sub(r'\b\d+[.,]?\d*\s*(g|kg|ml|l|đ|%|k)\b', '', name, flags=re.IGNORECASE).strip()
    if len(name) > 60: return ''
    return name


def predict_brand_product(ocr_text: str) -> tuple:
    """Simple rule-based pipeline (no ML). Used as tier 1."""
    brand, line = extract_brand_product(ocr_text)
    return normalize_brand(brand), normalize_product(line)


tests = [
    ("HALONG CANFOCO PATE CỘT ĐÈN HẢI PHÒNG",     ('Ha Long Canfoco', 'Pate Cột Đèn')),
    ("Công TY CP Đồ hộp Hạ Long bị bắt",         ('Đồ Hộp Hạ Long', '')),
    ("DO HOP HA LONG ISO 22000",                   ('Đồ Hộp Hạ Long', '')),
    ("PATE COT DEN CUA DO HOP HA LONG",            ('Đồ Hộp Hạ Long', '')),
    ("Sữa tươi tiệt trùng Vinamilk Flex 180ml",    ('Vinamilk', 'Flex')),
    ("Vinamilk EST 1976 thông báo khẩn",           ('Vinamilk', '')),
    ("NESTLÉ MILO Chocolate Malt Drink 3in1",      ('Nestlé Milo', '3in1')),
    ("Vissan PATE HEO 170g combo 3 hộp",           ('Vissan', 'Pate Heo')),
    ("Dutch Lady Grow+ 900g",                      ('Dutch Lady', 'Grow+')),
    ("Ba Vì Gold 1L",                              ('Ba Vì', 'Gold')),
    ("NAN OPTIPROPLUS thu hồi sữa",                ('Nestlé NAN OPTIpro PLUS', '')),
    ("LỜI NHẮN NHỦ VỀ NHÂN QUẢ VÀ DI SẢN",        ('', '')),
    ("b3 ifffÙ 200v set ị PIN",                    ('', '')),
    ("LS",                                          ('', '')),
]
passed = 0
print("Brand/Product split rules self-test:")
for text, exp in tests:
    got = predict_brand_product(text)
    ok  = got == exp
    passed += ok
    print(f"  {'PASS' if ok else 'FAIL'} '{text[:48]}' -> {got}" + ('' if ok else f"  (expected: {exp})"))
print(f"  {passed}/{len(tests)} passed | Brand rules: {len(BRAND_RULES)} | Line rules: {sum(len(v) for v in LINE_RULES.values())}")


## Cell 3b — TF-IDF Predictor for Brand_name & Product_name


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import time

try:
    from rapidfuzz import process, fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    _HAS_RAPIDFUZZ = False
    print("rapidfuzz not installed - fuzzy fallback will be skipped. Install with: pip install rapidfuzz")


def _derive_brand_line(row):
    brand, _ = extract_brand_product(row["ocr_text"])
    line = ""
    if brand and brand not in _BRAND_HAS_BUILTIN_LINE:
        combined = f'{row["ocr_text"]} {row["product_name"]}'
        line = _find_line(brand, combined.lower(), _strip_accent(combined))
    return pd.Series({"brand_derived": brand, "line_derived": line})


class NamePredictor:
    """Two stages: (1) binary classifier for has-label/no-label, (2) multi-class
    classifier predicting the specific name (brand or line) - shares the same
    architecture for both brand_name and product_name(line)."""

    def __init__(self, prob_threshold=0.45, max_features=15000, min_class_count=2):
        self.prob_threshold  = prob_threshold
        self.max_features    = max_features
        self.min_class_count = min_class_count
        self._has_clf = self._name_clf = None

    def fit(self, texts, labels):
        texts  = pd.Series(texts).astype(str).str.strip()
        labels = pd.Series(labels).astype(str).str.strip()
        tf = dict(analyzer="char_wb", ngram_range=(2,4),
                  max_features=self.max_features, min_df=2, sublinear_tf=True)
        lr = dict(max_iter=500, class_weight="balanced", C=2.0)

        self._has_clf = Pipeline([("t", TfidfVectorizer(**tf)),
                                   ("c", LogisticRegression(**lr))])
        self._has_clf.fit(texts, (labels != "").astype(int))

        pos  = pd.DataFrame({"text": texts, "label": labels})
        pos  = pos[(pos["text"] != "") & (pos["label"] != "")]
        keep = pos["label"].value_counts()
        pos  = pos[pos["label"].isin(keep[keep >= self.min_class_count].index)]

        self._name_clf = None
        if len(pos) and pos["label"].nunique() >= 2:
            self._name_clf = Pipeline([("t", TfidfVectorizer(**tf)),
                                        ("c", LogisticRegression(**lr))])
            self._name_clf.fit(pos["text"], pos["label"])
        return self

    def predict(self, text: str) -> str:
        text = "" if text is None else str(text).strip()
        if not text or not self._has_clf: return ""
        proba   = self._has_clf.predict_proba([text])[0]
        classes = list(self._has_clf.classes_)
        if 1 not in classes or proba[classes.index(1)] < self.prob_threshold: return ""
        if not self._name_clf: return ""
        return str(self._name_clf.predict([text])[0])


_KNOWN_BRANDS = sorted(set(b for _, b in BRAND_RULES))
_KNOWN_LINES  = sorted(set(l for rules in LINE_RULES.values() for _, l in rules))

def fuzzy_match_fallback(text: str, candidates, score_cutoff: int = 82) -> str:
    """Last resort fallback: approximate (typo-tolerant) matching."""
    if not _HAS_RAPIDFUZZ or not text or not text.strip() or not candidates:
        return ""
    match = process.extractOne(
        text, candidates, scorer=fuzz.partial_ratio, score_cutoff=score_cutoff
    )
    return match[0] if match else ""


brand_predictor = None
line_predictor  = None

if train_labels_df is not None:
    tl_df = train_labels_df.copy()
    tl_df["ocr_text"]     = tl_df["ocr_text"].astype(str).str.strip()
    tl_df["product_name"] = tl_df["product_name"].astype(str).str.strip()

    derived = tl_df.apply(_derive_brand_line, axis=1)
    tl_df   = pd.concat([tl_df, derived], axis=1)

    brand_predictor = NamePredictor()
    brand_predictor.fit(tl_df["ocr_text"], tl_df["brand_derived"])

    line_predictor = NamePredictor(min_class_count=2)
    line_predictor.fit(tl_df["ocr_text"], tl_df["line_derived"])

    n_brand = (tl_df["brand_derived"] != "").sum()
    n_line  = (tl_df["line_derived"]  != "").sum()
    print(f"Trained \u2713  |  brand labels: {n_brand:,}  |  line labels: {n_line:,}")
else:
    print("No train_labels available - using rules only")


def predict_brand_name(ocr_text: str) -> str:
    """3 tiers for brand_name: rule (highest precision) -> ML -> fuzzy fallback."""
    brand, _ = extract_brand_product(ocr_text)
    if not brand and brand_predictor is not None:
        brand = brand_predictor.predict(ocr_text)
    if not brand:
        brand = fuzzy_match_fallback(ocr_text, _KNOWN_BRANDS)
    return normalize_brand(brand)


def predict_product_name(ocr_text: str, brand: str = "") -> str:
    """3 tiers for product_name (line): rule -> ML -> fuzzy fallback.
    Brands that already have a built-in sub-line (NAN OPTIpro..., Pate Cot Den HP,
    Do Hop Ha Long) always get an empty product_name - no further lookup."""
    if brand in _BRAND_HAS_BUILTIN_LINE:
        return ""
    _, line = extract_brand_product(ocr_text)
    if not line and line_predictor is not None:
        line = line_predictor.predict(ocr_text)
    if not line and brand:
        line = fuzzy_match_fallback(ocr_text, _KNOWN_LINES)
    return normalize_product(line)


def predict_brand_product(ocr_text: str) -> tuple:
    """Combined function: returns (brand_name, product_name) after all 3 tiers."""
    brand = predict_brand_name(ocr_text)
    prod  = predict_product_name(ocr_text, brand)
    return brand, prod


if train_labels_df is not None:
    t0 = time.perf_counter()
    for _ in range(200): predict_brand_product("Vinamilk Flex 180ml")
    print(f"Inference: {(time.perf_counter()-t0)*1000/200:.3f} ms/img")


## Cell 3c — VnRestorer: Diacritic Restoration

In [ ]:
import re as _re
from unicodedata import normalize as _unorm2

_VN_RE = _re.compile(r'[\u00C0-\u024F\u1EA0-\u1EF9]')

def _no_acc(s: str) -> str:
    s = _unorm2('NFD', s)
    s = _re.sub(r'[\u0300-\u036f]', '', s)
    return _unorm2('NFC', s.replace('đ','d').replace('Đ','D')).lower()

def _alpha_count(t): return len(_re.findall(r'[a-zA-Z\u00C0-\u024F\u1EA0-\u1EF9]', t))
def _vn_count(t):    return len(_VN_RE.findall(t))
def _vn_ratio(t):
    if not t: return 0.0
    return _vn_count(t) / max(_alpha_count(t), 1)

_PHRASES = [
    ("doihoacitnhattu30namdoivoi", "đề nghị tù hoặc ít nhất tù 30 năm đối với"),
    ("itnhattu30nam",           "ít nhất tù 30 năm"),
    ("vidung130tanthitban",     "vì dùng 130 tấn thịt bẩn"),
    ("vuangan130tanthitlonbenh","vựa ngán 130 tấn thịt lợn bệnh"),
    ("130tanthitlonbenh",       "130 tấn thịt lợn bệnh"),
    ("thudoanhopthu",           "thủ đoạn hợp thức"),
    ("choagaysoc",              "choáng gây sốc"),
    ("tieuhuysanphamchebientuthitban", "tiêu hủy sản phẩm chế biến từ thịt bẩn"),
    ("sanphamchebientuthitban", "sản phẩm chế biến từ thịt bẩn"),
    ("chebientuthitban",        "chế biến từ thịt bẩn"),
    ("loanphaipatechebien",     "loạn phải pate chế biến"),
    ("patechebien",             "pate chế biến"),
    ("tuthitlonnhiembenh",      "từ thịt lợn nhiễm bệnh"),
    ("thitlonnhiembenh",        "thịt lợn nhiễm bệnh"),
    ("thitlonbenh",             "thịt lợn bệnh"),
    ("moinguytiemankhi",        "mối nguy tiềm ẩn khi"),
    ("bankhoekhong",            "bạn khỏe không"),
    ("suthatvethitthu",         "sự thật về thịt thối"),
    ("doilotdacsan",            "đội lốt đặc sản"),
    ("gocnhinnguoidan",         "góc nhìn người dân"),
    ("gocnhin",                 "góc nhìn"),
    ("nguoidan",                "người dân"),
    ("congdongmangdangquantamnhat", "cộng đồng mạng đang quan tâm nhất"),
    ("congdongmang",            "cộng đồng mạng"),
    ("dangquantam",              "đang quan tâm"),
    ("quantamnhat",              "quan tâm nhất"),
    ("noicjngdongmang",          "nơi cộng đồng mạng"),

    ("batkhancaptongiamdocvanhanviencongty", "bắt khẩn cấp tổng giám đốc và nhân viên công ty"),
    ("batkhancaptgdvanhanviencongty", "bắt khẩn cấp TGĐ và nhân viên công ty"),
    ("batkhancaptong",           "bắt khẩn cấp tổng"),
    ("batkhancap",                "bắt khẩn cấp"),
    ("tonggiamdoc",               "tổng giám đốc"),
    ("vanhanvien",                "và nhân viên"),
    ("phechuan",                  "phê chuẩn"),
    ("vksndtphaiphong",           "VKSND TP Hải Phòng"),

    ("congtycp",                  "Công ty CP"),

    ("banhmypatecotden",          "bánh mì pate cột đèn"),
    ("banhmypate",                "bánh mì pate"),
    ("patecotden",                "pate cột đèn"),
    ("patcotden",                 "pate cột đèn"),
    ("hatthuanchay",              "hạt thuần chay"),
    ("thuanchay",                 "thuần chay"),

    ("chauphinemphong4kho",       "châu Phi niêm phong 4 kho"),
    ("dichtachaup",               "dịch tả châu P"),

    ("highlandscoffeekhangdinhkhongsudung", "Highlands Coffee khẳng định không sử dụng"),
    ("khangdinhkhongsudung",      "khẳng định không sử dụng"),
    ("batkysanphamthit",          "bất kỳ sản phẩm thịt"),
    ("heochebiennaocuact",        "heo chế biến nào của CT"),
    ("highlandscoffeengungbantrasenvang", "Highlands Coffee ngừng bán trà sen vàng"),
    ("highlandscoffeengungban",   "Highlands Coffee ngừng bán"),
    ("ngungbantrasenvang",        "ngừng bán trà sen vàng"),
    ("trasenvang",                "trà sen vàng"),
    ("ngungban",                  "ngừng bán"),
    ("cungcap",                   "cung cấp"),
    ("codongthaimoi",             "có động thái mới"),
    ("nghivan",                   "nghi vấn"),

    ("uudaithanhto",              "ưu đãi thành tố"),
    ("banhngonmienphi",           "bánh ngon miễn phí"),
    ("mienphi",                   "miễn phí"),
    ("ketnoiniemtin",             "kết nối niềm tin"),
    ("bitancong",                 "bị tấn công"),
    ("lumcaptocgiamdoc",          "lùm xùm cấp tốc giám đốc"),
    ("dedontet",                  "để đón tết"),

    ("tam dung san xuat", "tạm dừng sản xuất"),
    ("san xuat",          "sản xuất"),
    ("san pham",          "sản phẩm"),
    ("thu hoi",           "thu hồi"),
    ("nguoi dung",        "người dùng"),
    ("nguoi tieu dung",   "người tiêu dùng"),
    ("khach hang",        "khách hàng"),
    ("cong ty",           "công ty"),
    ("co phan",           "cổ phần"),
    ("phat hien",         "phát hiện"),
    ("bi bat",            "bị bắt"),
    ("vu an",             "vụ án"),
    ("kiem tra",          "kiểm tra"),
    ("chat luong",        "chất lượng"),
    ("bao bi",            "bao bì"),
    ("nha san xuat",      "nhà sản xuất"),
    ("hang hoa",          "hàng hóa"),
    ("thi truong",        "thị trường"),
    ("an toan",           "an toàn"),
    ("suc khoe",          "sức khỏe"),
    ("dinh duong",        "dinh dưỡng"),
    ("tre em",            "trẻ em"),
    ("sua bot",           "sữa bột"),
    ("sua tuoi",          "sữa tươi"),
    ("thuc pham",         "thực phẩm"),
    ("nuoc mam",          "nước mắm"),
    ("banh mi",           "bánh mì"),
    ("ca phe",            "cà phê"),
    ("viet nam",          "Việt Nam"),
    ("ha noi",            "Hà Nội"),
    ("ho chi minh",       "Hồ Chí Minh"),
    ("da nang",           "Đà Nẵng"),
    ("ngay dang",         "ngày đăng"),
    ("nha hang",          "nhà hàng"),
    ("dia chi",           "địa chỉ"),
    ("gia re",            "giá rẻ"),
    ("chinh sach",        "chính sách"),
    ("doi voi",           "đối với"),
    ("hoac",              "hoặc"),
]
_PHRASES.sort(key=lambda kv: len(kv[0]), reverse=True)

def _augment_from_train(labels_df):
    extra = []
    if labels_df is None: return extra
    seen = {k for k,_ in _PHRASES}
    for col in ('ocr_text','product_name'):
        for raw in labels_df[col].astype(str).dropna():
            raw = raw.strip()
            if len(raw) < 6: continue
            if not _re.search(r'[\u00C0-\u024F\u1EA0-\u1EF9]', raw): continue
            words = raw.split()
            for n in (3,2):
                for i in range(len(words)-n+1):
                    phrase = ' '.join(words[i:i+n])
                    if not _re.search(r'[\u00C0-\u024F\u1EA0-\u1EF9]', phrase):
                        continue
                    key = _no_acc(phrase)
                    BRAND_KEYS = ("canfoco","canfood","ha long","halong","nan ",
                                  "milo","vinamilk","aptamil","highlands","vissan",
                                  "do hop","dohop","hop ha","hopha","pate",
                                  "nestle","dutch lady","th true","hipp","friso")
                    BRAND_STRINGS = ("halongcanfoco","dohophalong","congtydohophalong",
                                  "vinamilk","nestle","aptamil","highlands","vissan",
                                  "dutchlady","thtrue","hipp","friso","nan")
                    key_compact = key.replace(" ", "")
                    if any(b in key for b in BRAND_KEYS):
                        continue
                    if any(key_compact in bs or bs in key_compact for bs in BRAND_STRINGS if len(key_compact) >= 4):
                        continue
                    if len(key) >= 6 and key not in seen:
                        seen.add(key)
                        extra.append((key, phrase))
    return extra

_PHRASES = _PHRASES + _augment_from_train(train_labels_df)
_PHRASES.sort(key=lambda kv: len(kv[0]), reverse=True)
print(f"VnRestorer: {len(_PHRASES):,} phrases (static + mined from train_labels)")

_COMPILED_PHRASES = []
for _src, _tgt in _PHRASES:
    if len(_src) < 3:
        continue
    _src_compact = _src.replace(' ', '')
    if len(_src_compact) >= 6:
        for _pat in {_src, _src_compact}:
            _COMPILED_PHRASES.append(
                (_re.compile(_re.escape(_pat), _re.IGNORECASE), _tgt)
            )
    else:
        _regex = _re.compile(
            r'(?<![\w\u00C0-\u024F\u1EA0-\u1EF9])' + _re.escape(_src) +
            r'(?![\w\u00C0-\u024F\u1EA0-\u1EF9])', _re.IGNORECASE)
        _COMPILED_PHRASES.append((_regex, _tgt))

print(f"VnRestorer: {len(_COMPILED_PHRASES):,} compiled patterns ready")


def restore_text(text: str) -> str:
    """Restores Vietnamese diacritics. Only activates when vn_ratio < 0.10.

    Uses a character list plus a "protected" mask so that later phrase matches
    cannot overwrite a region already fixed by an earlier, longer phrase
    (avoids issues like "Long Canfoco" being corrupted by a shorter mined
    phrase "long can").
    """
    if not text: return text
    if _vn_ratio(text) >= 0.10:
        return text

    chars = list(text)
    protected = [False] * len(chars)

    def _apply(regex, tgt):
        nonlocal chars, protected
        result = ''.join(chars)
        out_chars, out_protected = [], []
        pos = 0
        for m in regex.finditer(result):
            st, en = m.start(), m.end()
            if any(protected[st:en]):
                continue
            out_chars.append(result[pos:st])
            out_protected.extend(protected[pos:st])
            out_chars.append(tgt)
            out_protected.extend([True] * len(tgt))
            pos = en
        out_chars.append(result[pos:])
        out_protected.extend(protected[pos:])
        chars = list(''.join(out_chars))
        protected = out_protected

    for regex, tgt in _COMPILED_PHRASES:
        _apply(regex, tgt)

    return ''.join(chars)


print("\nVnRestorer test:")
for txt in [
    "tam dung san xuat",
    "GOCNHINO NGUOIDAN PATECOTDEN Hạ Long SUTHATVETHITTHU DOILOTDACSAN",
    "Nestlé NAN OPTIpro PLUS",
    "HALONGCANFOCO pate",
]:
    r = restore_text(txt)
    print(f"  '{txt[:60]}'")
    print(f"  -> '{r[:60]}'")


## Cell 4 — OCR Engine 


In [ ]:
import time, numpy as np
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(
    ocr_version="PP-OCRv4",
    use_angle_cls=True,
    lang="vi",
    use_gpu=False,
    show_log=False,
    det_db_score_mode="slow",
    rec_batch_num=8,
)
print("PaddleOCR: PP-OCRv4 lang=ch (full Unicode, best for Vietnamese) \u2713")

def load_and_prep(image_id: str, max_dim=1280, min_dim=480):
    path = IMAGES_DIR / f"{image_id}.jpg"
    if not path.exists(): return None
    try:
        img = Image.open(path).convert("RGB")
    except Exception: return None
    w, h = img.size
    if max(w, h) < min_dim:
        s = min_dim / max(w, h)
        img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)
    elif max(w, h) > max_dim:
        s = max_dim / max(w, h)
        img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)
    img = ImageEnhance.Sharpness(img).enhance(1.8)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img

_OCR_FIXES = [
    (r'\b[Vv]inam[i1l][l1]k\b',    'Vinamilk'),
    (r'\b[Cc]anf[uo][ck][oa]\b',   'Canfoco'),
    (r'\bTH.?Tru[e3]\b',           'TH True'),
    (r'\b[Nn]estl[e3é]\b',         'Nestlé'),
    (r'\b[Mm][i1]lo\b',            'Milo'),
    (r'\b[Hh]ighland[s]?\b',       'Highlands'),
    (r'\b[Hh][aà].?[Ll][o0]ng\b',  'Hạ Long'),
    (r'\b[Nn][Aa][Nn]\b',          'NAN'),
    (r'\b[Aa]ptam[i1]l\b',         'Aptamil'),
    (r'\b[Hh][Ii][Pp][Pp]\b',      'HiPP'),
    (r'\b[Vv]issan\b',             'Vissan'),
    (r'(?<=[a-z])0(?=[a-z])',        'o'),
]

def _fix(text: str) -> str:
    for pat, rep in _OCR_FIXES:
        text = re.sub(pat, rep, text, flags=re.IGNORECASE)
    return text

def _dedup(text: str) -> str:
    toks = text.split()
    if not toks: return ''
    out = [toks[0]]
    for t in toks[1:]:
        if t.lower() != out[-1].lower(): out.append(t)
    return ' '.join(out)

def postprocess_ocr(text: str) -> str:
    text = re.sub(r'[\n\t\r]', ' ', text)
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1EA0-\u1EF9.,;:!?()/%%\-@#]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = _fix(text)
    text = restore_text(text)
    text = _dedup(text)
    return text[:500]

def _parse(result, thresh=0.30):
    lines = []
    if not result or not result[0]: return lines
    for line in result[0]:
        if not line or len(line) < 2: continue
        ti = line[1]
        if isinstance(ti, (list, tuple)) and len(ti) >= 2:
            txt, score = str(ti[0]).strip(), float(ti[1])
            if score >= thresh and txt:
                lines.append(txt)
    return lines

def _is_noise(raw: str) -> bool:
    if not raw: return True
    toks = raw.split()
    if len(toks) <= 1 and len(raw) < 4: return True
    alpha = sum(1 for t in toks if re.search(r'[a-zA-Z\u00C0-\u024F\u1EA0-\u1EF9]', t))
    return alpha / max(len(toks), 1) < 0.20

def run_ocr(image_id: str) -> tuple:
    """Returns (ocr_text, brand_name, product_name) - 3 separate values
    per the new spec (brand_name and product_name kept apart, not merged)."""
    img = load_and_prep(image_id)
    if img is None: return '', '', ''
    try:
        result = ocr_engine.ocr(np.array(img), cls=True)
        lines  = _parse(result, thresh=0.30)
        raw    = ' '.join(lines)
    except Exception:
        raw = ''

    if _is_noise(raw):
        return '', '', ''

    ocr_text = postprocess_ocr(raw)
    brand_name, product_name = predict_brand_product(ocr_text)
    return ocr_text, brand_name, product_name

print("\nSmoke test on first 5 images:")
for img_id in test_df['image_id'].iloc[:5]:
    t0 = time.time()
    ocr, brand, prod = run_ocr(img_id)
    vn = len(_VN_RE.findall(ocr))
    status = 'OK' if ocr else 'EMPTY'
    print(f"  [{status}] {img_id} ({time.time()-t0:.1f}s) | VN={vn}")
    if ocr:
        print(f"    ocr    : {ocr[:90]}")
        print(f"    brand  : {brand or '(empty)'}")
        print(f"    product: {prod or '(empty)'}")


## Cell 5 — Main Loop

In [ ]:
import os

if CHECKPOINT_CSV.exists(): os.remove(CHECKPOINT_CSV)

SAVE_EVERY = 25
done_ids, results = set(), []

if CHECKPOINT_CSV.exists():
    ckpt     = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt["image_id"])
    results  = ckpt.to_dict("records")
    print(f"Resuming: {len(done_ids):,} already done")
else:
    print("Starting fresh")

pending = [i for i in test_df["image_id"] if i not in done_ids]
print(f"Remaining: {len(pending):,} | Done: {len(done_ids):,}")

ocr_empty = 0
for idx, img_id in enumerate(tqdm(pending, desc="OCR")):
    ocr_text, brand_name, product_name = run_ocr(img_id)
    if not ocr_text and (IMAGES_DIR / f"{img_id}.jpg").exists():
        ocr_empty += 1
    results.append({
        "image_id": img_id,
        "ocr_text": ocr_text,
        "brand_name": brand_name,
        "product_name": product_name,
    })
    if (idx+1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")
    if (idx+1) % 250 == 0:
        df_tmp = pd.DataFrame(results)
        of = (df_tmp["ocr_text"].str.strip()!="").mean()
        bf = (df_tmp["brand_name"].str.strip()!="").mean()
        pf = (df_tmp["product_name"].str.strip()!="").mean()
        print(f"[{idx+1}] {img_id} | OCR fill {of:.0%} | brand fill {bf:.0%} | product fill {pf:.0%}")

pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")
df_r = pd.DataFrame(results)
print(f"\nProcessed   : {len(df_r):,}")
print(f"OCR fill    : {(df_r['ocr_text'].str.strip()!='').mean():.1%}  <- should be >= 70%")
print(f"Brand fill  : {(df_r['brand_name'].str.strip()!='').mean():.1%}")
print(f"Product fill: {(df_r['product_name'].str.strip()!='').mean():.1%}")
print(f"OCR empty   : {ocr_empty:,}  <- acceptable if the image genuinely has no text")


## Cell 6 — Export submission.csv

In [ ]:
import csv

sub    = pd.DataFrame(results)[["image_id","ocr_text","brand_name","product_name"]]
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
exp    = set(sample["image_id"]); got = set(sub["image_id"])

checks = {
    "Row count"    : len(sub)==len(sample),
    "No extra IDs" : len(got-exp)==0,
    "No missing"   : len(exp-got)==0,
    "No duplicates": not sub["image_id"].duplicated().any(),
    "No nulls"     : not sub.isnull().any().any(),
    "Columns OK"   : list(sub.columns)==["image_id","ocr_text","brand_name","product_name"],
}
all_pass = all(checks.values())
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

if all_pass:
    sub = sub.set_index("image_id").reindex(sample["image_id"]).reset_index()
    for col in ("ocr_text","brand_name","product_name"):
        sub[col] = sub[col].fillna("").astype(str).str.strip()
    out = sub.copy()
    for col in ("ocr_text","brand_name","product_name"):
        out.loc[out[col]=="", col] = " "
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
    print(f"\nSaved: {OUTPUT_CSV}")
    print(f"OCR fill    : {(sub['ocr_text'].str.strip()!='').mean():.1%}")
    print(f"Brand fill  : {(sub['brand_name'].str.strip()!='').mean():.1%}")
    print(f"Product fill: {(sub['product_name'].str.strip()!='').mean():.1%}")
    display(sub.head())
else:
    print("\nFAIL - fix errors before submitting")


## Cell 7 — Local Score (Score = 0.4×F1_brand + 0.35×(1−CER) + 0.25×F1_product)


In [ ]:
def composite_score(sol, sub, id_col="image_id"):
    """Score per the new format from the organizers:
        Score = 0.4 x F1_brand + 0.35 x (1 - CER) + 0.25 x F1_product

    - F1_brand   : token-level F1 matching brand_name (GT vs pred)
    - CER        : Character Error Rate on ocr_text (Levenshtein / len(gt))
    - F1_product : token-level F1 matching product_name (GT vs pred)

    Special cases (applied per component, not just the total):
      - GT empty & pred empty  -> that component = 1.0 (not penalized)
      - GT has a label & pred empty (or vice versa) -> that component = 0.0
    """
    def clean(v): return "" if pd.isna(v) else str(v).strip()

    m = sol.merge(sub, on=id_col, suffixes=("_gt", "_pred"))

    def f1(gt, pred):
        gt, pred = clean(gt), clean(pred)
        if not gt and not pred: return 1.0          # Case D: both empty -> full score
        g, p = set(gt.lower().split()), set(pred.lower().split())
        if not g or not p: return 0.0                # Case C: one side empty -> 0
        c = g & p
        pr, rc = len(c) / len(p), len(c) / len(g)
        return 0.0 if pr + rc == 0 else 2 * pr * rc / (pr + rc)

    def cer(gt, pred):
        gt, pred = clean(gt), clean(pred)
        if not gt and not pred: return 0.0           # Case D: both empty -> CER=0 -> (1-CER)=1.0
        if not gt: return 1.0                        # GT empty but pred not empty -> max CER
        if not pred: return 1.0                       # Case C: pred empty when GT has a label -> max CER
        m2, n = len(gt), len(pred)
        dp = list(range(n + 1))
        for i in range(1, m2 + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, n + 1):
                tmp = dp[j]
                dp[j] = prev if gt[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
                prev = tmp
        return min(dp[n] / len(gt), 1.0)

    f1_brand   = m.apply(lambda r: f1( r["brand_name_gt"],   r["brand_name_pred"]),   axis=1)
    f1_product = m.apply(lambda r: f1( r["product_name_gt"], r["product_name_pred"]), axis=1)
    cers       = m.apply(lambda r: cer(r["ocr_text_gt"],     r["ocr_text_pred"]),     axis=1)

    per_image = 0.4 * f1_brand + 0.35 * (1 - cers) + 0.25 * f1_product

    print(f"  F1_brand   (mean) : {f1_brand.mean():.4f}")
    print(f"  1-CER      (mean) : {(1 - cers).mean():.4f}")
    print(f"  F1_product (mean) : {f1_product.mean():.4f}")

    return round(per_image.mean(), 4)


def _t(gt_brand, gt_ocr, gt_prod, pr_brand, pr_ocr, pr_prod):
    sol  = pd.DataFrame([{"image_id":"x","ocr_text":gt_ocr,"brand_name":gt_brand,"product_name":gt_prod}])
    pred = pd.DataFrame([{"image_id":"x","ocr_text":pr_ocr,"brand_name":pr_brand,"product_name":pr_prod}])
    return composite_score(sol, pred)

print("Case A (all correct):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       "Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết"))
print("\nCase B (brand correct, product/OCR missing words):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       "Dove","Dove Smoothie","Smoothie"))
print("\nCase C (submitted empty when GT has a label):")
print("  Score =", _t("Dove","Dove Smoothie tẩy da chết","Smoothie tẩy da chết",
                       " "," "," "))
print("\nCase D (both GT and pred empty):")
print("  Score =", _t(""," ","", " ", " ", " "))


for sp in [Path("smce_dataset/solution.csv"), Path("/kaggle/input/smce-solution/solution.csv")]:
    if sp.exists():
        sol  = pd.read_csv(sp, keep_default_na=False)
        pred = pd.DataFrame(results)[["image_id","ocr_text","brand_name","product_name"]]
        print(f"\nLocal Score: {composite_score(sol, pred):.4f}")
        break
else:
    print("\nsolution.csv not found - expected for participants")
